# Test SWOT Lake Data Collection
Testing the `swot_file_collection_local_passes.py` script step by step.

## 1. Imports & Authentication

In [1]:
import os
import earthaccess
import pandas as pd

# Authenticate with Earthdata
auth = earthaccess.login(persist=True)
print(f"Authenticated: {auth.authenticated}")

c:\Users\xiaoye\anaconda3\envs\py311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Authenticated: True


## 2. Load SWOT Pass Numbers

In [2]:
swot_passes_path = r"E:\Project_2025_2026\Smart_hs\raw_data\swot\mekong_river_basin\swot_passes.csv"

swot_passes_df = pd.read_csv(swot_passes_path)
pass_names = swot_passes_df['pass_number'].values

print(f"Number of passes: {len(pass_names)}")
print(f"Pass numbers: {pass_names}")
swot_passes_df.head()

Number of passes: 48
Pass numbers: [  6   8  21  34  49  62  77  90 105 118 133 146 161 174 189 202 215 217
 230 243 258 271 284 286 299 312 327 340 355 368 383 396 411 424 439 452
 467 480 493 495 508 521 523 536 549 562 564 577]


,pass_number,geometry
0,6,"POLYGON ((31.916308 78.272068, 33.327768 78.26..."
1,8,"POLYGON ((6.025897 78.272068, 7.437357 78.2685..."
2,21,"POLYGON ((17.724587 -77.053767, 18.682245 -77...."
3,34,"POLYGON ((29.450555 78.272068, 30.862014 78.26..."
4,49,"POLYGON ((15.258834 -77.053767, 16.216491 -77...."


## 3. Test Search — Single Pass, Version C

In [5]:
# Use the first pass for a quick test
test_pass = 271
start_date = '2024-01-01'
end_date = '2024-02-01'

print(f"Testing pass: {test_pass}, period: {start_date} → {end_date}")

swot_results_version_c = earthaccess.search_data(
    short_name='SWOT_L2_HR_LakeSP_2.0',
    temporal=(start_date, end_date),
    granule_name=f'*Lake*_{test_pass}_AS*',
    count=-1
)

print(f"Version C results: {len(swot_results_version_c)}")

Testing pass: 271, period: 2024-01-01 → 2024-02-01
Version C results: 6


## 4. Test Search — Single Pass, Version D

In [6]:
swot_results_version_d = earthaccess.search_data(
    short_name='SWOT_L2_HR_LakeSP_D',
    temporal=(start_date, end_date),
    granule_name=f'*Lake*_{test_pass}_AS*',
    count=-1
)

print(f"Version D results: {len(swot_results_version_d)}")

Version D results: 3


## 5. Inspect a Single Granule's Metadata

In [12]:
all_results = list(swot_results_version_c) + list(swot_results_version_d)

if not all_results:
    print("No results found. Try a different pass number or date range.")
else:
    sample = all_results[4]
    umm = sample.get('umm', {})

    file_name_raw = umm.get('GranuleUR')
    file_name = file_name_raw[:-5] + '.zip'
    url = sample.data_links()[0]

    temporal = umm.get('TemporalExtent', {}).get('RangeDateTime', {})
    begin_dt = pd.to_datetime(temporal.get('BeginningDateTime'))
    end_dt   = pd.to_datetime(temporal.get('EndingDateTime'))

    print(f"GranuleUR  : {file_name_raw}")
    print(f"File name  : {file_name}")
    print(f"URL        : {url}")
    print(f"Begin time : {begin_dt}")
    print(f"End time   : {end_dt}")
    print(f"All links  : {sample.data_links()}")

GranuleUR  : SWOT_L2_HR_LakeSP_Obs_009_271_AS_20240113T193324_20240113T194703_PGC0_01
File name  : SWOT_L2_HR_LakeSP_Obs_009_271_AS_20240113T193324_20240113T194703_PG.zip
URL        : https://archive.swot.podaac.earthdata.nasa.gov/podaac-swot-ops-cumulus-protected/SWOT_L2_HR_LakeSP_2.0/SWOT_L2_HR_LakeSP_Obs_009_271_AS_20240113T193324_20240113T194703_PGC0_01.zip
Begin time : 2024-01-13 19:33:24.921000+00:00
End time   : 2024-01-13 19:47:03.959000+00:00
All links  : ['https://archive.swot.podaac.earthdata.nasa.gov/podaac-swot-ops-cumulus-protected/SWOT_L2_HR_LakeSP_2.0/SWOT_L2_HR_LakeSP_Obs_009_271_AS_20240113T193324_20240113T194703_PGC0_01.zip']


## 6. Build DataFrame for All Results (Single Pass)

In [8]:
records = []

for result in all_results:
    umm = result.get('umm', {})
    temporal = umm.get('TemporalExtent', {}).get('RangeDateTime', {})
    begin_dt = pd.to_datetime(temporal.get('BeginningDateTime'))
    end_dt   = pd.to_datetime(temporal.get('EndingDateTime'))
    url = result.data_links()[0]

    records.append({
        'year':       begin_dt.year,
        'month':      begin_dt.month,
        'day':        begin_dt.day,
        'start_date': begin_dt,
        'end_date':   end_dt,
        'url':        url,
    })

swot_file_df = pd.DataFrame(records)
print(f"Total records: {len(swot_file_df)}")
swot_file_df

Total records: 9


,year,month,day,start_date,end_date,url
0,2024,1,13,2024-01-13 19:33:24.921000+00:00,2024-01-13 19:47:03.959000+00:00,https://archive.swot.podaac.earthdata.nasa.gov...
1,2024,1,13,2024-01-13 19:33:24.921000+00:00,2024-01-13 19:47:03.959000+00:00,https://archive.swot.podaac.earthdata.nasa.gov...
2,2024,1,13,2024-01-13 19:33:24.921000+00:00,2024-01-13 19:47:03.959000+00:00,https://archive.swot.podaac.earthdata.nasa.gov...
3,2024,1,13,2024-01-13 19:33:24.921000+00:00,2024-01-13 19:47:03.959000+00:00,https://archive.swot.podaac.earthdata.nasa.gov...
4,2024,1,13,2024-01-13 19:33:24.921000+00:00,2024-01-13 19:47:03.959000+00:00,https://archive.swot.podaac.earthdata.nasa.gov...
5,2024,1,13,2024-01-13 19:33:24.921000+00:00,2024-01-13 19:47:03.959000+00:00,https://archive.swot.podaac.earthdata.nasa.gov...
6,2024,1,13,2024-01-13 19:33:24.921000+00:00,2024-01-13 19:47:03.959000+00:00,https://archive.swot.podaac.earthdata.nasa.gov...
7,2024,1,13,2024-01-13 19:33:24.921000+00:00,2024-01-13 19:47:03.959000+00:00,https://archive.swot.podaac.earthdata.nasa.gov...
8,2024,1,13,2024-01-13 19:33:24.921000+00:00,2024-01-13 19:47:03.959000+00:00,https://archive.swot.podaac.earthdata.nasa.gov...


## 7. Run Full Collection for All Passes (Single Month)

In [13]:
all_records = []

for pass_name in pass_names:
    print(f"Searching pass {pass_name}...", end=' ')

    for short_name, version_label in [
        ('SWOT_L2_HR_LakeSP_2.0', 'C'),
        ('SWOT_L2_HR_LakeSP_D',   'D'),
    ]:
        results = earthaccess.search_data(
            short_name=short_name,
            temporal=(start_date, end_date),
            granule_name=f'*Lake*_{pass_name}_AS*',
            count=-1
        )
        for r in results:
            umm = r.get('umm', {})
            temporal_r = umm.get('TemporalExtent', {}).get('RangeDateTime', {})
            begin_dt = pd.to_datetime(temporal_r.get('BeginningDateTime'))
            end_dt   = pd.to_datetime(temporal_r.get('EndingDateTime'))
            all_records.append({
                'pass':       pass_name,
                'version':    version_label,
                'year':       begin_dt.year,
                'month':      begin_dt.month,
                'day':        begin_dt.day,
                'start_date': begin_dt,
                'end_date':   end_dt,
                'url':        r.data_links()[0],
            })

    print(f"{len([x for x in all_records if x['pass'] == pass_name])} records")

full_df = pd.DataFrame(all_records)
print(f"\nTotal records across all passes: {len(full_df)}")
full_df

Searching pass 6... 0 records
Searching pass 8... 0 records
Searching pass 21... 

KeyboardInterrupt: 

## 8. Save Results

In [ ]:
save_folder = r"E:\Project_2025_2026\Smart_hs\raw_data\swot\mekong_river_basin\swot_lakes\file_list"
os.makedirs(save_folder, exist_ok=True)

out_path = os.path.join(save_folder, f'{start_date}_{end_date}_swot_lake_file_df.csv')
full_df.to_csv(out_path, index=False)
print(f"Saved {len(full_df)} records to:\n  {out_path}")